# Quantum patch classifier, Task 3.1

Compares a VQC (6 qubits, amplitude encoding, 2 layers) against a classical SVM on
Raw, CLAHE, MSRCR and UDCP inputs.

Images without a mask are only used for inference, never to make up metrics. The SAM
masks are pseudo-labels until a human checks them, and they should trace the pipe, not
whatever object SAM happens to latch onto. This notebook doesn't touch Dataset_Final_512
or the mask-free augmentations.

In [ ]:
%pip -q install pennylane torch gdown seaborn scikit-learn opencv-python-headless pandas joblib


In [ ]:
from pathlib import Path
import json, re, time, shutil, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2, gdown, torch, pennylane as qml, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, jaccard_score, confusion_matrix
torch.set_num_threads(2)
torch.set_default_dtype(torch.float64)
ROOT=Path("/content/quantum_patch_classifier")
DATA=ROOT/"data"
OUT=ROOT/("results_"+time.strftime("%Y%m%d_%H%M%S"))
OUT.mkdir(parents=True,exist_ok=True)
METHODS=["Crudos","CLAHE","MSRCR","UDCP"]
SEEDS=[17,42,101]
SIZE,PATCH,BATCH,EPOCHS=64,8,64,25
LR=.03
INVERT_MASKS=False
BACKGROUND_MAX,STRUCTURE_MIN=.05,.25
print("PennyLane:",qml.__version__,"| resultados:",OUT)


## 1. Download

The Drive inventory is built in and we download file by file by ID, because gdown chokes
on the folder-level limit. It's a snapshot: anything added to Drive later won't show up
on its own. If Drive blocks access or we hit the quota the run stops instead of limping
on with half a dataset.

In [ ]:
MANIFEST={
 "CLAHE": [
  {
   "id": "1fgDd2ePY84hz5fJqCxn3LCe_azrLiDSo",
   "name": "DeepWater-Video7_frame_0009.jpg"
  },
  {
   "id": "1knw3Wv3sDnuJehAC_8puU9jTte_1hEDr",
   "name": "DeepWater-Video7_frame_0008.jpg"
  },
  {
   "id": "1mcYPBxq4DdBCIoCxahcM4CHxzX-A7m5i",
   "name": "DeepWater-Video7_frame_0007.jpg"
  },
  {
   "id": "1oS34TsRkVoVrJZo_jmcFWeQdaam0QGNJ",
   "name": "DeepWater-Video7_frame_0006.jpg"
  },
  {
   "id": "1APHELeXCvOPQCjQi_vU_qrgF3gRPpDFv",
   "name": "DeepWater-Video7_frame_0005.jpg"
  },
  {
   "id": "187Poe2FupHQM9TBOZIAvbVGAqWgc3cX4",
   "name": "DeepWater-Video7_frame_0004.jpg"
  },
  {
   "id": "1xjmEnOACiW-uXFW4SztTNzMMrt-_etCH",
   "name": "DeepWater-Video7_frame_0003.jpg"
  },
  {
   "id": "1qwj9L-zYBlQRqpUwYTRXY6NpR6baKXER",
   "name": "DeepWater-Video7_frame_0002.jpg"
  },
  {
   "id": "1rlyaYfCA35gwv8-sYMjkcS5FEMOx1XzN",
   "name": "DeepWater-Video7_frame_0001.jpg"
  },
  {
   "id": "1xb4YiJOh79Nj4y4oVdLJqGWDiv-6W2Qb",
   "name": "DeepWater-Video7_frame_0000.jpg"
  },
  {
   "id": "1l-_2J7nvTvrB4J9mISfe2OjyijoTHIqw",
   "name": "DeepWater-Video6_frame_0023.jpg"
  },
  {
   "id": "1DDCc70I6ax9B0L5QO5w1g9d73ceGJjLE",
   "name": "DeepWater-Video6_frame_0022.jpg"
  },
  {
   "id": "1osGy1FBZAP3aAvC1YTAHlEUkglEAbAfM",
   "name": "DeepWater-Video6_frame_0021.jpg"
  },
  {
   "id": "16Q4ZEKd4gFjGpgjzlFnbLc_8P5qfK949",
   "name": "DeepWater-Video6_frame_0020.jpg"
  },
  {
   "id": "1xHhapiz5YVakF-7Rx4uWamzjEwa6-Xb-",
   "name": "DeepWater-Video6_frame_0019.jpg"
  },
  {
   "id": "1Ycz8mc2ojnPqbkdsS7BXkRDO11jpEJBc",
   "name": "DeepWater-Video6_frame_0018.jpg"
  },
  {
   "id": "1bp3Tzb9TP6NLXj63zjuZX6Ns7RkM4BNH",
   "name": "DeepWater-Video6_frame_0017.jpg"
  },
  {
   "id": "1OLiwWnTWeo9AFhg70HY7DFwhtbAeQ0yw",
   "name": "DeepWater-Video6_frame_0016.jpg"
  },
  {
   "id": "131kAobM-kW7D3Dr6uNWQLrgAeFdYdQzD",
   "name": "DeepWater-Video6_frame_0015.jpg"
  },
  {
   "id": "1xwMhlJ3U6wjYeeWkGMfy50h19vgtlfdz",
   "name": "DeepWater-Video6_frame_0014.jpg"
  },
  {
   "id": "1hGtMx4GgpvDmhzfPkKT4MnRITDvQ8u5T",
   "name": "DeepWater-Video6_frame_0013.jpg"
  },
  {
   "id": "1iB3QIbZdHC5M7nfg3dk3AcLmj_FEdFSM",
   "name": "DeepWater-Video6_frame_0012.jpg"
  },
  {
   "id": "12Go-uVisePOxyRTDNWaofGjI40csf2lG",
   "name": "DeepWater-Video6_frame_0011.jpg"
  },
  {
   "id": "1VM5Of8Pffb5ZbYXeA5l8lMIsTeRLxqAU",
   "name": "DeepWater-Video6_frame_0010.jpg"
  },
  {
   "id": "1sbThjJyCJXkI9sxLFwCVvaNuBiIpeNrc",
   "name": "DeepWater-Video6_frame_0009.jpg"
  },
  {
   "id": "1NtOevajFPz5ah7AqJgJJ7wihBvQArOao",
   "name": "DeepWater-Video6_frame_0008.jpg"
  },
  {
   "id": "1-xKD-MbvLcfAPkdT4XSYPnQIaIt8R9HJ",
   "name": "DeepWater-Video6_frame_0007.jpg"
  },
  {
   "id": "1Dr7PumjAvgbMbix7EBhHGuZDDOQnksAE",
   "name": "DeepWater-Video6_frame_0006.jpg"
  },
  {
   "id": "1RR1rdlm57okcm_V-ly8MDn9DF5v_bXKn",
   "name": "DeepWater-Video6_frame_0005.jpg"
  },
  {
   "id": "1CO51EbeqMXJXqZ-AB-o66E8SWrp2wYCG",
   "name": "DeepWater-Video6_frame_0004.jpg"
  },
  {
   "id": "1vLt5xqkgPT2_czy94KpHB1aJevRiM3o0",
   "name": "DeepWater-Video6_frame_0003.jpg"
  },
  {
   "id": "1ePT18nLxo9mNRU15gQ-3vUyjHTOH0B_s",
   "name": "DeepWater-Video6_frame_0002.jpg"
  },
  {
   "id": "129ePsg_ZUdrR81EhtczRDooOufpcPcvH",
   "name": "DeepWater-Video6_frame_0001.jpg"
  },
  {
   "id": "1YJPWlf-Hmd_w4fVH5mUOAu2mrCARb4_W",
   "name": "DeepWater-Video6_frame_0000.jpg"
  },
  {
   "id": "1J-Eyu6jBCJEcYk65paSwyapA9QtBJhLQ",
   "name": "DeepWater-Video3_frame_0048.jpg"
  },
  {
   "id": "1MNiiNSFpioUTex3DZWOzfGfuoTGMyBXf",
   "name": "DeepWater-Video3_frame_0047.jpg"
  },
  {
   "id": "1tX4DGDX9q-Fg_qMChrCvGAIIcIyZUWRW",
   "name": "DeepWater-Video3_frame_0046.jpg"
  },
  {
   "id": "1-_X7k0FS9-b2E6o5YjRYRIZNjPdwtsfz",
   "name": "DeepWater-Video3_frame_0045.jpg"
  },
  {
   "id": "1dYoEeeBewAzN4xC2IONfjoKbCa6zOPqG",
   "name": "DeepWater-Video3_frame_0044.jpg"
  },
  {
   "id": "1y42bW18fw4IiZgqNV1tQLnwxsfeZ-kEf",
   "name": "DeepWater-Video3_frame_0043.jpg"
  },
  {
   "id": "1Th1xpHyz_TwCUDR8sA99Hu9Z6FBmTXDP",
   "name": "DeepWater-Video3_frame_0042.jpg"
  },
  {
   "id": "1yZ-Y0AvYmRJASZUN7-obwYR0vGNr5UML",
   "name": "DeepWater-Video3_frame_0041.jpg"
  },
  {
   "id": "1xHrw92tJ_4d_2KWB_PwaoEwzISqqfIgR",
   "name": "DeepWater-Video3_frame_0040.jpg"
  },
  {
   "id": "1E3r6swCkMTC-3kMHqa4b7__CRJfnOZTE",
   "name": "DeepWater-Video3_frame_0039.jpg"
  },
  {
   "id": "1pL88VjQmDrymrzDEJRG6Bll1B7diuVQ_",
   "name": "DeepWater-Video3_frame_0038.jpg"
  },
  {
   "id": "1MOWQGUC21AbeDsaezKP1Xf-jqwwZrANV",
   "name": "DeepWater-Video3_frame_0037.jpg"
  },
  {
   "id": "1hSRhKiHaX6DFwfgXMzcLNdm73B236vGO",
   "name": "DeepWater-Video3_frame_0036.jpg"
  },
  {
   "id": "1HAF2gS0MPEeCvBPsglgK-We7nwKDXatD",
   "name": "DeepWater-Video3_frame_0035.jpg"
  },
  {
   "id": "1IlBYVEloKCYpcj72ndpyJIpsOWRd5kMk",
   "name": "DeepWater-Video3_frame_0034.jpg"
  },
  {
   "id": "13JCJ31DnvXZuAF-QzG2DtcbuHrwI4RXW",
   "name": "DeepWater-Video3_frame_0033.jpg"
  },
  {
   "id": "1FHHJRGuiJyEIlzDjorWpj_UsCy5yL9VU",
   "name": "DeepWater-Video3_frame_0032.jpg"
  },
  {
   "id": "1u3l6KzKQQS5qL9Zh8mEX2w_oOGjmf4nK",
   "name": "DeepWater-Video3_frame_0031.jpg"
  },
  {
   "id": "1O9Bow2tNp4HFanLsinxjL5f-zfZqRbLd",
   "name": "DeepWater-Video3_frame_0030.jpg"
  },
  {
   "id": "1XCCz6dsF6smdrALErmbSy0mzmcg3MTfb",
   "name": "DeepWater-Video3_frame_0029.jpg"
  },
  {
   "id": "1KnYdqyUXmx6NfcWZOGM8Pz4RjjxmlEzl",
   "name": "DeepWater-Video3_frame_0028.jpg"
  },
  {
   "id": "1cKB68MZFCI1J5pPhfpr1n4sS_vpWxtlc",
   "name": "DeepWater-Video3_frame_0027.jpg"
  },
  {
   "id": "1MWPQs9sHitYr7bXNUO0u2PRvCFPGn3KA",
   "name": "DeepWater-Video3_frame_0026.jpg"
  },
  {
   "id": "1XEvymdehM0E-hzBZEBhl3pbQXMeTQfIL",
   "name": "DeepWater-Video3_frame_0025.jpg"
  },
  {
   "id": "1k2v22CfngxiJarwb-yPQCJBfeYd679ZU",
   "name": "DeepWater-Video3_frame_0024.jpg"
  },
  {
   "id": "1C7CgOBdyCRZALEwJP6zCTQN2nZO0LjBG",
   "name": "DeepWater-Video3_frame_0023.jpg"
  },
  {
   "id": "1a3_RLSAg2AeZlPwfNd3cEK_qkJ87TKFe",
   "name": "DeepWater-Video3_frame_0022.jpg"
  },
  {
   "id": "1uaRmLIWdPdHs9dVfgPY0XSkid1duKEL_",
   "name": "DeepWater-Video3_frame_0021.jpg"
  },
  {
   "id": "1j8t8dYLh4naS1nZXQaELIZKspyGt1GZd",
   "name": "DeepWater-Video3_frame_0020.jpg"
  },
  {
   "id": "1_J579CTarl0T--fw76sqX5oTFW4GW2A9",
   "name": "DeepWater-Video3_frame_0019.jpg"
  },
  {
   "id": "1NIKis4qRZTuDSwVfOnK1IYsW6avn5Cj-",
   "name": "DeepWater-Video3_frame_0018.jpg"
  },
  {
   "id": "1mQc-hSTyRi-A5QiN8RHNgD3usXWGV-L6",
   "name": "DeepWater-Video3_frame_0017.jpg"
  },
  {
   "id": "1AW11Ofrgk7Eiicf8RBwNbvcRzolMLhW-",
   "name": "DeepWater-Video3_frame_0016.jpg"
  },
  {
   "id": "1FGFYN0r2hjF3WXMppshAXEM8PuJP1KaK",
   "name": "DeepWater-Video3_frame_0015.jpg"
  },
  {
   "id": "10l4FUjVnh3KleZj6tt7R2whLKoop9bG5",
   "name": "DeepWater-Video3_frame_0014.jpg"
  },
  {
   "id": "1T5JRpKkpzlCiYr6u10tJMdg1b8rM5xZV",
   "name": "DeepWater-Video3_frame_0013.jpg"
  },
  {
   "id": "1H48RmW3MogFjJApPlcmVZwMVWYtqW2Av",
   "name": "DeepWater-Video3_frame_0012.jpg"
  },
  {
   "id": "1dcupn-7UpTQkZz-vlQbejqsDpK5H2Aie",
   "name": "DeepWater-Video3_frame_0011.jpg"
  },
  {
   "id": "1WSkgv33WSWBWfS6xeAaWR5JqeZS_5fzo",
   "name": "DeepWater-Video3_frame_0010.jpg"
  },
  {
   "id": "1tqntt-rZjwhjFYr09qm1TFcUMoFhCM6T",
   "name": "DeepWater-Video3_frame_0009.jpg"
  },
  {
   "id": "1tBDhKp7FyREorrEb314Jj_YUAz_w0wWi",
   "name": "DeepWater-Video3_frame_0008.jpg"
  },
  {
   "id": "18AIvYqg_pe6KtxRjeyaYSHeX8L2W856q",
   "name": "DeepWater-Video3_frame_0007.jpg"
  },
  {
   "id": "1pvaHSjQuMq_wUm766mmDbo63mSWUjg7W",
   "name": "DeepWater-Video3_frame_0006.jpg"
  },
  {
   "id": "1hoeNVTokbPlCFBswy-Hj3hikM-4zPrPq",
   "name": "DeepWater-Video3_frame_0005.jpg"
  },
  {
   "id": "1GNnCM1Yf4wenmOHbTygvW1WtYfDYUz8t",
   "name": "DeepWater-Video3_frame_0004.jpg"
  },
  {
   "id": "1yrTMgbjEl-CSnBG_3cVXfguHB8v15UXW",
   "name": "DeepWater-Video3_frame_0003.jpg"
  },
  {
   "id": "1mJ-j12kdjRwujVvdtH8dn-JDmJ_LKhL6",
   "name": "DeepWater-Video3_frame_0002.jpg"
  },
  {
   "id": "1goGdSxxI6UB1W5S6X_dWWPZyQZF4xhDF",
   "name": "DeepWater-Video3_frame_0001.jpg"
  },
  {
   "id": "18JK_Yq0fxCWLywqohvBN95G2XdnjUyGL",
   "name": "DeepWater-Video3_frame_0000.jpg"
  },
  {
   "id": "1vbqIWvXumDsgFC_jT9kyvD2E56POHo-q",
   "name": "DeepWater-Video1_frame_0021.jpg"
  },
  {
   "id": "1N3PdPsRYS8SSHrE5VCKlzc53PBY8z8T7",
   "name": "DeepWater-Video1_frame_0020.jpg"
  },
  {
   "id": "1xjLUbCwWMv1rIUcgviv5hGmNa4-_nhiv",
   "name": "DeepWater-Video1_frame_0019.jpg"
  },
  {
   "id": "1nIwFRiVMiLodsUGvLWumc4T-K0KThFFW",
   "name": "DeepWater-Video1_frame_0018.jpg"
  },
  {
   "id": "1uefSBqnSUjyNISczCGgFSvCGZScYwi1l",
   "name": "DeepWater-Video1_frame_0017.jpg"
  },
  {
   "id": "1hjk2W_WpK1-bSo6SeHo3lfXmazLbF8mK",
   "name": "DeepWater-Video1_frame_0016.jpg"
  },
  {
   "id": "182DWIYjEVF_X64JzlFdTj99o1cML1lid",
   "name": "DeepWater-Video1_frame_0015.jpg"
  },
  {
   "id": "1ZEkmRxvZU4Qxx8P14oYxP2uhx2d0snPL",
   "name": "DeepWater-Video1_frame_0014.jpg"
  },
  {
   "id": "1O-XKeX2-0fb_pjVOcQRfWJGzna8d7oo2",
   "name": "DeepWater-Video1_frame_0013.jpg"
  },
  {
   "id": "17ALlJ0uaZu2o-kVN5RgKc9_XKJhWCITt",
   "name": "DeepWater-Video1_frame_0012.jpg"
  },
  {
   "id": "11K610NL-yWC1SkpOD42Vv1MEL3TLH7Je",
   "name": "DeepWater-Video1_frame_0011.jpg"
  },
  {
   "id": "1n9zmXwQs7O3mBMikVoGSpmqcQfx6TkDH",
   "name": "DeepWater-Video1_frame_0010.jpg"
  },
  {
   "id": "1eZN2XC5ZkvPPIXelL5X3XhKiTcKTb29p",
   "name": "DeepWater-Video1_frame_0009.jpg"
  },
  {
   "id": "1Ua9gcECM3Lr-SwaWoK5xgxMHnxmNRXPE",
   "name": "DeepWater-Video1_frame_0008.jpg"
  },
  {
   "id": "1bV5dir7f65iZwiq8MLpWD497LCLCcGyx",
   "name": "DeepWater-Video1_frame_0007.jpg"
  },
  {
   "id": "1L7AHhSO7HezZSDpJq_LAfnWR0Lt1zaRP",
   "name": "DeepWater-Video1_frame_0006.jpg"
  },
  {
   "id": "1DPkyTT77_PD_X9TFyWFZSr7V_oInhYuN",
   "name": "DeepWater-Video1_frame_0005.jpg"
  },
  {
   "id": "1Zpb0i92sgBjV7lbUVgGjbYQSpl5YdvoZ",
   "name": "DeepWater-Video1_frame_0004.jpg"
  },
  {
   "id": "1BPx_adll5CyZcGTjG9Yt4Qaq2uTI0DHd",
   "name": "DeepWater-Video1_frame_0003.jpg"
  },
  {
   "id": "1hD0g9s8P-To8KqdceYk_EKYM2Zqch2Y-",
   "name": "DeepWater-Video1_frame_0002.jpg"
  },
  {
   "id": "1qiNv8AvTOJj_ybFNfzvqSI70y49L-00L",
   "name": "DeepWater-Video1_frame_0001.jpg"
  },
  {
   "id": "1TTGoVspb7WBP07TZUCQd5zkCk9jt9160",
   "name": "DeepWater-Video1_frame_0000.jpg"
  }
 ],
 "Crudos": [
  {
   "id": "1OywoanTlpuvq9JUQHW0rd4ZkIxVfXLme",
   "name": "DeepWater-Video3_frame_0048.jpg"
  },
  {
   "id": "1M03MZMLa9iZtjRbISCPuHAY5IDc_vk5F",
   "name": "DeepWater-Video3_frame_0047.jpg"
  },
  {
   "id": "1NbttKiIMhUx9ZqDW5psBgcQ3xiTSPyta",
   "name": "DeepWater-Video3_frame_0046.jpg"
  },
  {
   "id": "1k-wakm1adYq0Gzm6H49CKOjJ2u8YNN3N",
   "name": "DeepWater-Video3_frame_0045.jpg"
  },
  {
   "id": "1VfTtY_I1YKpqdBuogBJyleVu8HMNmwzi",
   "name": "DeepWater-Video3_frame_0044.jpg"
  },
  {
   "id": "1g39_bGMED45d2y8GBcufzhXpqXq3GYlo",
   "name": "DeepWater-Video3_frame_0043.jpg"
  },
  {
   "id": "1xy-C9x6BZtCZg_3idevaig1-PrNoG4Bt",
   "name": "DeepWater-Video3_frame_0042.jpg"
  },
  {
   "id": "1eDaMCLXfTrZ5GkdZrVtPZ9oHNk9iDdbg",
   "name": "DeepWater-Video3_frame_0041.jpg"
  },
  {
   "id": "11iPQzGuLa-9Lmx-udJxtQx_z0N1J9cyW",
   "name": "DeepWater-Video3_frame_0040.jpg"
  },
  {
   "id": "1K7Aw5SG5kT3nOZ-EA-QBocrxlE0bG6Hy",
   "name": "DeepWater-Video3_frame_0039.jpg"
  },
  {
   "id": "1OdP7CxZGsKzMtkQ6BZtGjgOATnnAXjKe",
   "name": "DeepWater-Video3_frame_0038.jpg"
  },
  {
   "id": "1VKqMNimqQne23LjfD_DigUdmnDbMRSs8",
   "name": "DeepWater-Video3_frame_0037.jpg"
  },
  {
   "id": "1q7V3hKPxpEbgI5f1U9VgnBuVBK2VdGNw",
   "name": "DeepWater-Video3_frame_0036.jpg"
  },
  {
   "id": "13uIo5ani5p8hftKNXAxp77CPFA0Dm5fQ",
   "name": "DeepWater-Video3_frame_0035.jpg"
  },
  {
   "id": "1kOmy5SH8TVs75Bq6D2md8IizwyrqNaBD",
   "name": "DeepWater-Video3_frame_0034.jpg"
  },
  {
   "id": "12nooJEM1X7r_wPuv3wNEGK1ej0GKU6Dw",
   "name": "DeepWater-Video3_frame_0033.jpg"
  },
  {
   "id": "1KRc9W3_xn1jHD1BfQryyqPZIkSYNHU6L",
   "name": "DeepWater-Video3_frame_0032.jpg"
  },
  {
   "id": "1S2llzOhAdGTAHXaPo4pHOWs5XN4bnEpU",
   "name": "DeepWater-Video3_frame_0031.jpg"
  },
  {
   "id": "1E7vpiK0vu1L73sQPUBlAEWAzHfgyAefC",
   "name": "DeepWater-Video3_frame_0030.jpg"
  },
  {
   "id": "1PARz-ocHl6pdHxsk8F9WtRMs7kQwuz1D",
   "name": "DeepWater-Video3_frame_0029.jpg"
  },
  {
   "id": "1BDhLCjITKm-pErkWpKd6tsWSnImgiUr8",
   "name": "DeepWater-Video3_frame_0028.jpg"
  },
  {
   "id": "17nCWQOUJvyiJM3dfcB6Z0dvdgDfEQoWZ",
   "name": "DeepWater-Video3_frame_0027.jpg"
  },
  {
   "id": "1ymA7AT5fNrbPS6LArPSZwLu5OjkwFtXK",
   "name": "DeepWater-Video3_frame_0026.jpg"
  },
  {
   "id": "1W2tI4c4DSHuqCo9f4iCBPXeygLz9U34y",
   "name": "DeepWater-Video3_frame_0025.jpg"
  },
  {
   "id": "12zetJ1Ys-6gPhmH138dxndZnV_gKO75x",
   "name": "DeepWater-Video3_frame_0024.jpg"
  },
  {
   "id": "1-2h7ToyELt0gX2LIcN_tBhR5eSvn-U1W",
   "name": "DeepWater-Video3_frame_0023.jpg"
  },
  {
   "id": "12yuD4zkQ7B7CmMnTR-b_QmmXxZ7d8-hR",
   "name": "DeepWater-Video3_frame_0022.jpg"
  },
  {
   "id": "1zYJQz-qmHzmgc-wYtOIEw4EjT0xZGwj-",
   "name": "DeepWater-Video3_frame_0021.jpg"
  },
  {
   "id": "1IXYRCA925tze3LhlBHhBxcH2WNcq-StC",
   "name": "DeepWater-Video3_frame_0020.jpg"
  },
  {
   "id": "1QRfvyrDbWIvv5dwtbnvC6AM7ZrrLjJU8",
   "name": "DeepWater-Video3_frame_0019.jpg"
  },
  {
   "id": "1leSzjsoPd_77NLevmqPEGxQcaKuOs3yG",
   "name": "DeepWater-Video3_frame_0018.jpg"
  },
  {
   "id": "1TiuJf8ONA4g3-KxQ0_PczYZbaJ0KT4M0",
   "name": "DeepWater-Video3_frame_0017.jpg"
  },
  {
   "id": "1s8oezqrKgkEk6zv1sDBbbXPLcMM7DoYF",
   "name": "DeepWater-Video3_frame_0016.jpg"
  },
  {
   "id": "1QF4lgMZNcGRj5daJ32BjWu5U0QNLuIeF",
   "name": "DeepWater-Video3_frame_0015.jpg"
  },
  {
   "id": "1_XsOm0TXQ3cRN-AP7Ma9pWoiw1MGAzYs",
   "name": "DeepWater-Video3_frame_0014.jpg"
  },
  {
   "id": "19n20zqYdMh5rUJFKiw-sP9hR3UH9c6ND",
   "name": "DeepWater-Video3_frame_0013.jpg"
  },
  {
   "id": "1tuVDlBewzhaN-EII0QyGZvEkkxzTdF3s",
   "name": "DeepWater-Video3_frame_0012.jpg"
  },
  {
   "id": "1h4_lkImUjqk7zevU_-2yvrIRriFOulFa",
   "name": "DeepWater-Video3_frame_0011.jpg"
  },
  {
   "id": "1m_pHZckjTFLHEfl6k9Et8DFPL_oZUssD",
   "name": "DeepWater-Video3_frame_0010.jpg"
  },
  {
   "id": "1mzX3ZYFTx00fL0sdbsFzLUS1djYJsvH7",
   "name": "DeepWater-Video3_frame_0009.jpg"
  },
  {
   "id": "1cpHKKWlmuxG0sYV4CQpc6_ihW3vzLSTG",
   "name": "DeepWater-Video3_frame_0008.jpg"
  },
  {
   "id": "1hcbdn9UT6oO8bLcAKWapqQRf0z2hX_jm",
   "name": "DeepWater-Video3_frame_0007.jpg"
  },
  {
   "id": "1UIUXxj89-A9Semt0U6iN1BG49h9yzdEh",
   "name": "DeepWater-Video3_frame_0006.jpg"
  },
  {
   "id": "1YQMzu7Gc3LdmaHtBTQNLB18iDIEYQbcB",
   "name": "DeepWater-Video3_frame_0005.jpg"
  },
  {
   "id": "1kBwBnuSWjSbO-cedg4kpWTmThGn_zx9U",
   "name": "DeepWater-Video3_frame_0004.jpg"
  },
  {
   "id": "1Kgp7oFc_vmd86sV99UM9vJ8j24CZURgv",
   "name": "DeepWater-Video3_frame_0003.jpg"
  },
  {
   "id": "1mhxhPiqjK2omwrzp1HGwJAfbK9_wJDXd",
   "name": "DeepWater-Video3_frame_0002.jpg"
  },
  {
   "id": "1Ka5y-XfNX7if3ah4HZKM-RIBtVN-N3HZ",
   "name": "DeepWater-Video3_frame_0001.jpg"
  },
  {
   "id": "1QhC5FR36RpqVdghL2X_Nxbws4tgKCQoV",
   "name": "DeepWater-Video3_frame_0000.jpg"
  },
  {
   "id": "1BUEBbTBDSKphPQW9X3qYR_Iv9oztEDSs",
   "name": "DeepWater-Video1_frame_0021.jpg"
  },
  {
   "id": "1pnXlOxMoznac4f2MYKF3CgFHNHrm2EWx",
   "name": "DeepWater-Video1_frame_0020.jpg"
  },
  {
   "id": "104MwBaq5zZPRnsEmMGMZbUJggKXNzN3c",
   "name": "DeepWater-Video1_frame_0019.jpg"
  },
  {
   "id": "1kyz6c3b9YBfp4BeciRGPBfyF9pwZ9FsH",
   "name": "DeepWater-Video1_frame_0018.jpg"
  },
  {
   "id": "11KSnvCSCQ3cisMnTvTBKYatMsX5DODGL",
   "name": "DeepWater-Video1_frame_0017.jpg"
  },
  {
   "id": "1mvbf7WblVGVxfa8ACM7ujF-0RxSqqGR1",
   "name": "DeepWater-Video1_frame_0016.jpg"
  },
  {
   "id": "1EHj65tD38am1fTKJQ5cR3znCtH_INZ_f",
   "name": "DeepWater-Video1_frame_0015.jpg"
  },
  {
   "id": "1f--TcZeTAG92iJAUZWrb09TIexDIFLEu",
   "name": "DeepWater-Video1_frame_0014.jpg"
  },
  {
   "id": "1gyI_L5FMp5v33ZPP0KFMqelxROQU5P1v",
   "name": "DeepWater-Video1_frame_0013.jpg"
  },
  {
   "id": "1Mp5OYhAalu53u-GWXNAOoZREEnEkAW4f",
   "name": "DeepWater-Video1_frame_0012.jpg"
  },
  {
   "id": "1h4fkzhPGfHtanQKnaefbySK86-f_10xc",
   "name": "DeepWater-Video1_frame_0011.jpg"
  },
  {
   "id": "1Td3sO8yzJypZy0IOW3FBm5_Xi8suVX1r",
   "name": "DeepWater-Video1_frame_0010.jpg"
  },
  {
   "id": "17d4XTzHhgLJlZxxksYzWuO9aS6WHGsGF",
   "name": "DeepWater-Video1_frame_0009.jpg"
  },
  {
   "id": "18zgHaiUpaUHTDhrn_ZfDAhkUBlad8zQ1",
   "name": "DeepWater-Video1_frame_0008.jpg"
  },
  {
   "id": "1aM4FVDsEIh48LOHhMcmH8Xn9oVatqQwV",
   "name": "DeepWater-Video1_frame_0007.jpg"
  },
  {
   "id": "1wnZUteGi7HOjB-ya58-oXzo_YZbsSvYW",
   "name": "DeepWater-Video1_frame_0006.jpg"
  },
  {
   "id": "14PsMCJ8TUmYW9kgq-_AOSOUqxAWNiQJh",
   "name": "DeepWater-Video1_frame_0005.jpg"
  },
  {
   "id": "1FK1lFBM3TEpNA328ujh9Q0iJYQAgpUUe",
   "name": "DeepWater-Video1_frame_0004.jpg"
  },
  {
   "id": "1nf4r1SocTd9d4cZdMKEffLj-FeEPbsP3",
   "name": "DeepWater-Video1_frame_0003.jpg"
  },
  {
   "id": "1zGsineLAOh7Vrak9dhGJUUqI6j6faPW6",
   "name": "DeepWater-Video1_frame_0002.jpg"
  },
  {
   "id": "1jODzJAkJq4mWqsLyJE-loU08dWm98zt8",
   "name": "DeepWater-Video1_frame_0001.jpg"
  },
  {
   "id": "1ZySxjXdkdYN-LyhQh9ePsBscJ5hDLwG2",
   "name": "DeepWater-Video1_frame_0000.jpg"
  },
  {
   "id": "1G0EXBM-xPgBgRaKddhMUzxXTv0XWKugC",
   "name": "DeepWater-Video6_frame_0023.jpg"
  },
  {
   "id": "1JlcJh-IKT5bO4WCGtvCB5n_UtGNDgsdu",
   "name": "DeepWater-Video6_frame_0022.jpg"
  },
  {
   "id": "1ScGXME9d9Xg-5jIHql9kN-myzHgu-aNy",
   "name": "DeepWater-Video6_frame_0021.jpg"
  },
  {
   "id": "1PRe9_EK--i8gr4RV2DOxURNOxti1vD2a",
   "name": "DeepWater-Video6_frame_0020.jpg"
  },
  {
   "id": "1FuOz_2RthbRRJtfF19ocVm6yDUlD0gkC",
   "name": "DeepWater-Video6_frame_0019.jpg"
  },
  {
   "id": "1brTx8WKtvS8pp-a1aDBIu9gZjxX_vJ3P",
   "name": "DeepWater-Video6_frame_0018.jpg"
  },
  {
   "id": "13IZhqw2n0q2f58et2kERK8oCDyl6Slzq",
   "name": "DeepWater-Video6_frame_0017.jpg"
  },
  {
   "id": "1qXsfLO9iP1zF9EhW7YTT6oZs6wqYKVgR",
   "name": "DeepWater-Video6_frame_0016.jpg"
  },
  {
   "id": "12lBEAIjUZGASHfn7ALulkq9TQ8ZnaCqZ",
   "name": "DeepWater-Video6_frame_0015.jpg"
  },
  {
   "id": "1v6bE-oRSpynkMx7vQBIKNb4z7Yvv0-4B",
   "name": "DeepWater-Video6_frame_0014.jpg"
  },
  {
   "id": "1nd2FegFZI0qr5VJ8CFafN7MlEXjLQAYq",
   "name": "DeepWater-Video6_frame_0013.jpg"
  },
  {
   "id": "1YDMRKS5ikzlNYVa1sUkKMrMFkmz2Nmb8",
   "name": "DeepWater-Video6_frame_0012.jpg"
  },
  {
   "id": "1barIh-XVC80S66Kpw3zGlEEQAOZOQl4B",
   "name": "DeepWater-Video6_frame_0011.jpg"
  },
  {
   "id": "1TTrcmaZ8TLaD6nyROczBrc96kUwQGvBN",
   "name": "DeepWater-Video6_frame_0010.jpg"
  },
  {
   "id": "1Aa8qBWS0__FnSjOL9jOMdV0iTyObVyll",
   "name": "DeepWater-Video6_frame_0009.jpg"
  },
  {
   "id": "1JoZbLMDQRl7bGCU-wMqLhgnqc8bv6y-1",
   "name": "DeepWater-Video6_frame_0008.jpg"
  },
  {
   "id": "1iSu2NysL4AhSVneNbn3oUfU6_rLIMAij",
   "name": "DeepWater-Video6_frame_0007.jpg"
  },
  {
   "id": "1skAstOmqfI-mvU8HMpmkMyrIGNZUerc9",
   "name": "DeepWater-Video6_frame_0006.jpg"
  },
  {
   "id": "161EGPFTQNOG6tW9WBlR_2M1zl0Q9_jQl",
   "name": "DeepWater-Video6_frame_0005.jpg"
  },
  {
   "id": "1XEO8Ozu93Wm51RW_9udGJMME4EZqGDuI",
   "name": "DeepWater-Video6_frame_0004.jpg"
  },
  {
   "id": "1ut5BrpgMycWEl06k5pNK5TZn4fn8eoNZ",
   "name": "DeepWater-Video6_frame_0003.jpg"
  },
  {
   "id": "1LEo5EnXC4ykJ7WVNBngpJgUFLhiYLIKZ",
   "name": "DeepWater-Video6_frame_0002.jpg"
  },
  {
   "id": "1Ai08T-z5NGhdU5IJkqO0cTeG4TZsxKJQ",
   "name": "DeepWater-Video6_frame_0001.jpg"
  },
  {
   "id": "1GWbMFbLd58Dt0krslp6l-OIGQbiDAEyn",
   "name": "DeepWater-Video6_frame_0000.jpg"
  },
  {
   "id": "12brTFO4tOwOY38ve_BpfEQn55E18LdFK",
   "name": "DeepWater-Video7_frame_0009.jpg"
  },
  {
   "id": "1yy4kANwvzcxn6ZDIbZVQ5Y26MwqUCOvS",
   "name": "DeepWater-Video7_frame_0008.jpg"
  },
  {
   "id": "1EEMH7GVo18t4719V1WndHRxFTz8H11xJ",
   "name": "DeepWater-Video7_frame_0007.jpg"
  },
  {
   "id": "1UZo5VyrtvKUe7YdsUJURAhh86TPSdNAL",
   "name": "DeepWater-Video7_frame_0006.jpg"
  },
  {
   "id": "1PhgScVroDeNnQeZp5i2tOKUSdKNi8vVu",
   "name": "DeepWater-Video7_frame_0005.jpg"
  },
  {
   "id": "1blP8qupsWhMSkEUvESM8gmSECc5oeNIv",
   "name": "DeepWater-Video7_frame_0004.jpg"
  },
  {
   "id": "17Mcgiq0qoGWshCSG2ln2ARWyZeg8xTLW",
   "name": "DeepWater-Video7_frame_0003.jpg"
  },
  {
   "id": "1TOIT3_inCCaGCTiUvRdUKOAXkXxvbsqr",
   "name": "DeepWater-Video7_frame_0002.jpg"
  },
  {
   "id": "1vunFJTmxYAj_ClxShKKE1sOF9kN16X-h",
   "name": "DeepWater-Video7_frame_0001.jpg"
  },
  {
   "id": "1-ATJufv237riBvr4Om1LYeYkKG_uf6ps",
   "name": "DeepWater-Video7_frame_0000.jpg"
  }
 ],
 "MSRCR": [
  {
   "id": "1p_Z_IzLM1UWFRNeHbXETOd6feJvZULEF",
   "name": "DeepWater-Video7_frame_0009.jpg"
  },
  {
   "id": "1vyn5VESdu_5lV5tp_lclWZZYinB1u0KA",
   "name": "DeepWater-Video7_frame_0008.jpg"
  },
  {
   "id": "1CdIwNq53YRi2Ndf57-WfxEy0u1S0C8_r",
   "name": "DeepWater-Video7_frame_0007.jpg"
  },
  {
   "id": "1pI3OzUGB9bhJU_2aUOdqYbTKuRlweyvz",
   "name": "DeepWater-Video7_frame_0006.jpg"
  },
  {
   "id": "1BDHhHKnPYmoPauF7hFIZTPpseiE_RLud",
   "name": "DeepWater-Video7_frame_0005.jpg"
  },
  {
   "id": "1yGJO2m0PVvwNcumU4hmIJQOsZ6GpCJ4v",
   "name": "DeepWater-Video7_frame_0004.jpg"
  },
  {
   "id": "1KucefBwJkKI51mNO0CayNxX4_yAKt-Gy",
   "name": "DeepWater-Video7_frame_0003.jpg"
  },
  {
   "id": "1VbK6Q4NRbJgNOvpIEMV4hL_BwzuSJ6Hh",
   "name": "DeepWater-Video7_frame_0002.jpg"
  },
  {
   "id": "1wteojhjvJOdkn8H9pArQC0D6z_L2abu8",
   "name": "DeepWater-Video7_frame_0001.jpg"
  },
  {
   "id": "1CCNySrgnqrJgkZZdNfmJrVT72BGQn1FR",
   "name": "DeepWater-Video7_frame_0000.jpg"
  },
  {
   "id": "1mRZDoLGo5G57r5e8TQ7GGCVrC_eyLg5U",
   "name": "DeepWater-Video6_frame_0023.jpg"
  },
  {
   "id": "1OpeK-oVgWfWDdfCfPSWXM47OxJU2D0cT",
   "name": "DeepWater-Video6_frame_0022.jpg"
  },
  {
   "id": "1oGhVKP_dtaTegmyXU12FZ1IcLcNw2Oml",
   "name": "DeepWater-Video6_frame_0021.jpg"
  },
  {
   "id": "1qvgA5HxzCCfRo793f_1KYJVSuNTQAmg0",
   "name": "DeepWater-Video6_frame_0020.jpg"
  },
  {
   "id": "1FIbhI9AuDNxNmUKf0WoyXvRN6pwW-VGk",
   "name": "DeepWater-Video6_frame_0019.jpg"
  },
  {
   "id": "1LL4UARZNmMFBnJa7bwIt-xCKo45Xuk_w",
   "name": "DeepWater-Video6_frame_0018.jpg"
  },
  {
   "id": "1K289_-CMUo0jLW6CvBeBIRFF1uILcDeY",
   "name": "DeepWater-Video6_frame_0017.jpg"
  },
  {
   "id": "1HkePamrkevr7znbDsRqAoRIGN2iXaLuE",
   "name": "DeepWater-Video6_frame_0016.jpg"
  },
  {
   "id": "18EMTCLdPCtDKz-40Y8U0x9olJo4-Mem5",
   "name": "DeepWater-Video6_frame_0015.jpg"
  },
  {
   "id": "1stDk_CBiPrHZXsgJHWdA59qKMNK98yxx",
   "name": "DeepWater-Video6_frame_0014.jpg"
  },
  {
   "id": "1LtCrZD3hy-4AMjEcKoU86Vf2npKsuJlz",
   "name": "DeepWater-Video6_frame_0013.jpg"
  },
  {
   "id": "16YRmFIm5pnChilD8YBlB0k3TlSDqIiOb",
   "name": "DeepWater-Video6_frame_0012.jpg"
  },
  {
   "id": "1xts9DJj_dgf0S_7hJjJqHeXOlh8Dg3eH",
   "name": "DeepWater-Video6_frame_0011.jpg"
  },
  {
   "id": "1xCs_z-pxHhYvjsKXL-i7g9AfNDiFW0bR",
   "name": "DeepWater-Video6_frame_0010.jpg"
  },
  {
   "id": "1E3nPE86cladfI21AN4Vty-ksIYdcVqaM",
   "name": "DeepWater-Video6_frame_0009.jpg"
  },
  {
   "id": "1n0w6uPt26xA-TRUE_1CvFC05WvUcZwON",
   "name": "DeepWater-Video6_frame_0008.jpg"
  },
  {
   "id": "1prRaYhFfRMrRveN2GZ_iKNvOqdqHlnJe",
   "name": "DeepWater-Video6_frame_0007.jpg"
  },
  {
   "id": "1xRr0JUQphv-NsuWQkN5VSYa-gIzpETen",
   "name": "DeepWater-Video6_frame_0006.jpg"
  },
  {
   "id": "14x0fvyFuhBwhRti8fi2SQH88GziEQkE1",
   "name": "DeepWater-Video6_frame_0005.jpg"
  },
  {
   "id": "1tV611h2rSL4zY8aY656o0s4i-ZQYavSw",
   "name": "DeepWater-Video6_frame_0004.jpg"
  },
  {
   "id": "1PfoNp93ByNTayh-aXoSTglGQ6QuFXE5K",
   "name": "DeepWater-Video6_frame_0003.jpg"
  },
  {
   "id": "1ItWtwcGO2sV_pKlR7CXcSmOpbBCK-LRd",
   "name": "DeepWater-Video6_frame_0002.jpg"
  },
  {
   "id": "13WSmo4SoGRORChfAT864moj2WCK3WS_T",
   "name": "DeepWater-Video6_frame_0001.jpg"
  },
  {
   "id": "16_6r8TDtj5V4lH016EwbAge5-bmKlLzW",
   "name": "DeepWater-Video6_frame_0000.jpg"
  },
  {
   "id": "1KTECYZIj5imyTRRmp9pTQzPgeD33-Wvp",
   "name": "DeepWater-Video3_frame_0048.jpg"
  },
  {
   "id": "1n4Saa3UjdMdGcoNsLrvoz2kYceg7SHBN",
   "name": "DeepWater-Video3_frame_0047.jpg"
  },
  {
   "id": "17T47tjD55Ep87NH7_XX4hhyspoHbPXqD",
   "name": "DeepWater-Video3_frame_0046.jpg"
  },
  {
   "id": "10jmyszGR7LPQuUMX3hLaoFHdOMVDfgWm",
   "name": "DeepWater-Video3_frame_0045.jpg"
  },
  {
   "id": "1pjivuu8KdhqOGu3xxQvSEYxf52Pauz06",
   "name": "DeepWater-Video3_frame_0044.jpg"
  },
  {
   "id": "1mWcX1QpcBXaVt0JrdQy6qscA6wF7UP2y",
   "name": "DeepWater-Video3_frame_0043.jpg"
  },
  {
   "id": "1It8NK9GPrnUopI2bD6VJ0gdK3tD0NCFK",
   "name": "DeepWater-Video3_frame_0042.jpg"
  },
  {
   "id": "1-Nx3I6y3AB1_Fuc-iLxAbvzXGwiLvWfh",
   "name": "DeepWater-Video3_frame_0041.jpg"
  },
  {
   "id": "1W7GhmGeTSWVuwcMY21nHD-MLQIQLio6M",
   "name": "DeepWater-Video3_frame_0040.jpg"
  },
  {
   "id": "1fUiB5Pu_9UaYT-t-kQWLsQm8Y826qjGQ",
   "name": "DeepWater-Video3_frame_0039.jpg"
  },
  {
   "id": "11tv1-53fdLaZXLWl8-eu2NoWtvKvDgs5",
   "name": "DeepWater-Video3_frame_0038.jpg"
  },
  {
   "id": "1G9MN7KFlr2TyUKKyDdMN-U1BQmi-j9dT",
   "name": "DeepWater-Video3_frame_0037.jpg"
  },
  {
   "id": "1QpzlndUA-nHaZGtCvoUUbYQqyef9Y6Pq",
   "name": "DeepWater-Video3_frame_0036.jpg"
  },
  {
   "id": "1McpJyi9vThrB_T_cp-nWCI_Ot0bmaoUL",
   "name": "DeepWater-Video3_frame_0035.jpg"
  },
  {
   "id": "1AMUlfOSxy7cTCtghigoqGJxaQJXTSNdB",
   "name": "DeepWater-Video3_frame_0034.jpg"
  },
  {
   "id": "1ghWbqTOEt4EZCvgRSRHTsZS5aB4Ixu0T",
   "name": "DeepWater-Video3_frame_0033.jpg"
  },
  {
   "id": "1FAmEPuif90Xnd2WffU9K68pHVpZ1j3IO",
   "name": "DeepWater-Video3_frame_0032.jpg"
  },
  {
   "id": "1HGRkT9Ly-yxRdUZkgzePu-E66Tlc8oPO",
   "name": "DeepWater-Video3_frame_0031.jpg"
  },
  {
   "id": "1HsdI52MBEoVk1QbTmtHBuNaY2aMrPieD",
   "name": "DeepWater-Video3_frame_0030.jpg"
  },
  {
   "id": "14TRYl3u-FWuofbmUA9Cf9zVXfXhqnluT",
   "name": "DeepWater-Video3_frame_0029.jpg"
  },
  {
   "id": "1rxMjdXJU4NDEAC7grSPD4UI8GyprSCGM",
   "name": "DeepWater-Video3_frame_0028.jpg"
  },
  {
   "id": "1NjM_33bm5tdTJM9SUPs8BWU6CZXgM7Jk",
   "name": "DeepWater-Video3_frame_0027.jpg"
  },
  {
   "id": "1rX9DsJF2frwqLBmqEJ0Be9a1xPsfd7wx",
   "name": "DeepWater-Video3_frame_0026.jpg"
  },
  {
   "id": "1Zmg55zCVdxUfZK0tAocFvaeZv6dKbYjr",
   "name": "DeepWater-Video3_frame_0025.jpg"
  },
  {
   "id": "1C2B04Zl0GYHfa_xIsDUPoP_X87givSVn",
   "name": "DeepWater-Video3_frame_0024.jpg"
  },
  {
   "id": "1WbvfpFUey4TnE9yxCtW8q2HEpgQCT6Iy",
   "name": "DeepWater-Video3_frame_0023.jpg"
  },
  {
   "id": "1ce_kVwIuQGSf7jC3MWyvE4WbFXc3RF01",
   "name": "DeepWater-Video3_frame_0022.jpg"
  },
  {
   "id": "18ZtUzIKcP8xTKIQJtXYVmBGn-y0g1Pui",
   "name": "DeepWater-Video3_frame_0021.jpg"
  },
  {
   "id": "1hI0SRqks6Tr5suz4z-KxOcZp0B3ghKlQ",
   "name": "DeepWater-Video3_frame_0020.jpg"
  },
  {
   "id": "1Wj95651kH5GGk7Ehx-ZdxOvfObnSPTBK",
   "name": "DeepWater-Video3_frame_0019.jpg"
  },
  {
   "id": "1269yyX2aE8B5cx7mPjuFBenzCiB3_b5J",
   "name": "DeepWater-Video3_frame_0018.jpg"
  },
  {
   "id": "1WcD4VUDSoI7GReH-o6RC3Sq7g8UJrZYh",
   "name": "DeepWater-Video3_frame_0017.jpg"
  },
  {
   "id": "1nKv35nPW1SuFea04ncEdnBcLz7cB3t0c",
   "name": "DeepWater-Video3_frame_0016.jpg"
  },
  {
   "id": "1HHDdcn_kcIk-oRhApB8S5zEWawZPgkFM",
   "name": "DeepWater-Video3_frame_0015.jpg"
  },
  {
   "id": "1xt5qAhzIWUrt9lwK2SSgZe29uOcvJbcB",
   "name": "DeepWater-Video3_frame_0014.jpg"
  },
  {
   "id": "1LHqPLEjMw3ZrYLMn0kcxWBROfsRk_4rH",
   "name": "DeepWater-Video3_frame_0013.jpg"
  },
  {
   "id": "1bd6RPVDcbR3RYv2X5dU1qIH4xa80Y13C",
   "name": "DeepWater-Video3_frame_0012.jpg"
  },
  {
   "id": "1SFySXlaWG_85mkdMYNYLTpVqQ2gEsd72",
   "name": "DeepWater-Video3_frame_0011.jpg"
  },
  {
   "id": "15YPn6PobBmdZF44RTmtBigQhYyJVuuX9",
   "name": "DeepWater-Video3_frame_0010.jpg"
  },
  {
   "id": "1M6c0mzk6bn_kJtfvo8h_g0lykzTDzdUw",
   "name": "DeepWater-Video3_frame_0009.jpg"
  },
  {
   "id": "1BoH2V_bFn5IjkfJ18-aJ1iRn1Yh2J4Gp",
   "name": "DeepWater-Video3_frame_0008.jpg"
  },
  {
   "id": "1n1bKcVGihALHiZoCY1fNej4AwvrI_zjQ",
   "name": "DeepWater-Video3_frame_0007.jpg"
  },
  {
   "id": "1WOj5Bv0JLM-ClgeciIjsJy_46afGjSl6",
   "name": "DeepWater-Video3_frame_0006.jpg"
  },
  {
   "id": "1jA_IKLVM7XnrOS8JNs2pLkvLQtWL67dm",
   "name": "DeepWater-Video3_frame_0005.jpg"
  },
  {
   "id": "1HdyBvvvD9PBSYNuOJXYCbqK8RTfyeeNs",
   "name": "DeepWater-Video3_frame_0004.jpg"
  },
  {
   "id": "1FNlPEkGGuqHqEaY_pVJygbDuV-ww2AP5",
   "name": "DeepWater-Video3_frame_0003.jpg"
  },
  {
   "id": "1xqGpuJJZyiziXOH6oDZFjOicqHYfm9Ze",
   "name": "DeepWater-Video3_frame_0002.jpg"
  },
  {
   "id": "1p6hIXA9NjJfbqlwnKIiAVIhWA9KhIq5H",
   "name": "DeepWater-Video3_frame_0001.jpg"
  },
  {
   "id": "133OvAH5P6iMK27sO-CT5SeXjzqoI43yi",
   "name": "DeepWater-Video3_frame_0000.jpg"
  },
  {
   "id": "1f0W2qeAucqeLjoBcrxw39_fvpsEnCoTl",
   "name": "DeepWater-Video1_frame_0021.jpg"
  },
  {
   "id": "1cf0R6LEbra7NEP7gflMmd4KY-Tynntvj",
   "name": "DeepWater-Video1_frame_0020.jpg"
  },
  {
   "id": "1R-XBi4bvG4GEBcRB0ewjkyLWlZjAdcj3",
   "name": "DeepWater-Video1_frame_0019.jpg"
  },
  {
   "id": "1oQ6T-dH9N4HpyopYy8L-KW4iPQxCtCnK",
   "name": "DeepWater-Video1_frame_0018.jpg"
  },
  {
   "id": "1Vbuq_SXBvJdnp7NE8AvH43sVrT1Bh6Xb",
   "name": "DeepWater-Video1_frame_0017.jpg"
  },
  {
   "id": "1QteNcCXxCW8V8Xx2aQe5sgSmxCnXC31C",
   "name": "DeepWater-Video1_frame_0016.jpg"
  },
  {
   "id": "18yryUxyEwchFggpNsdxEG3sopT7X5uHA",
   "name": "DeepWater-Video1_frame_0015.jpg"
  },
  {
   "id": "1X41pi1A_kXPdR1VooqJTSGTEeTlaASZU",
   "name": "DeepWater-Video1_frame_0014.jpg"
  },
  {
   "id": "16tgWGnNG0KSG5O3HIu_jFOXyrS8wsW13",
   "name": "DeepWater-Video1_frame_0013.jpg"
  },
  {
   "id": "1I8fyd-B0Yk_1DYjpJUkYeHv6573NSwDb",
   "name": "DeepWater-Video1_frame_0012.jpg"
  },
  {
   "id": "1u_oath6EP8W-OBH7HDq1UmBXWkZAe2Co",
   "name": "DeepWater-Video1_frame_0011.jpg"
  },
  {
   "id": "1-5abGe-rEb6YdtjdUYogHWDRztb-Lrmm",
   "name": "DeepWater-Video1_frame_0010.jpg"
  },
  {
   "id": "11hvh-25w0ZD8Q-ccSQwPB8IIFmE_AR3-",
   "name": "DeepWater-Video1_frame_0009.jpg"
  },
  {
   "id": "1r1XkJuVQjFUn1Nlzxekj7FDfLKK4B1l8",
   "name": "DeepWater-Video1_frame_0008.jpg"
  },
  {
   "id": "1AF6P1DkqhVL_iXsd-bsidi8huaLmSLca",
   "name": "DeepWater-Video1_frame_0007.jpg"
  },
  {
   "id": "11yGtVAnskh9IQ6Aw5wtBw8WSuiv_yhpL",
   "name": "DeepWater-Video1_frame_0006.jpg"
  },
  {
   "id": "1Flro2w4KvS3zWZMca5TbU4Y4rQ8aftpo",
   "name": "DeepWater-Video1_frame_0005.jpg"
  },
  {
   "id": "13i25MZeeWBFoLEO6fljZDhDLuK5aIdn9",
   "name": "DeepWater-Video1_frame_0004.jpg"
  },
  {
   "id": "1MiApHVAizyAJoQgz2G-7OnNgG2V4lw5P",
   "name": "DeepWater-Video1_frame_0003.jpg"
  },
  {
   "id": "18OrXBrckwVd5QPTF8vP8MEG7QUg9hygv",
   "name": "DeepWater-Video1_frame_0002.jpg"
  },
  {
   "id": "12FA72wxerCHXajo_u-_7gthwY7cGgGcl",
   "name": "DeepWater-Video1_frame_0001.jpg"
  },
  {
   "id": "1nvpOwPAwcANxgPjdr-VvMHRD0eK3btwY",
   "name": "DeepWater-Video1_frame_0000.jpg"
  }
 ],
 "Masks": [
  {
   "id": "1w6UQ9n1ohrTOtWuOyxICmdz96Xs1a7JF",
   "name": "DeepWater-Video1_frame_0019_mask.png"
  },
  {
   "id": "1v7a3Dnts2HVE_ArULzM8-3hkFxJbHHgr",
   "name": "DeepWater-Video1_frame_0018_mask.png"
  },
  {
   "id": "1l_tTReRmAfYL6V0qEy8uDqPfX_Y3Ii1N",
   "name": "DeepWater-Video1_frame_0017_mask.png"
  },
  {
   "id": "1IqVPN_5zNH8VgxV3xbAYO5DY2fYlsO8K",
   "name": "DeepWater-Video1_frame_0016_mask.png"
  },
  {
   "id": "1cwilZqIYLXMLir41tAnIq0PnVSaJQNlZ",
   "name": "DeepWater-Video1_frame_0015_mask.png"
  },
  {
   "id": "1Pgv0MF4ILk_IZ74_dZYMpX33X-oIsmgx",
   "name": "DeepWater-Video1_frame_0014_mask.png"
  },
  {
   "id": "14_kEReDxYwMeXzbT2ikCRIuYg4rOIGnu",
   "name": "DeepWater-Video1_frame_0013_mask.png"
  },
  {
   "id": "111FY32r7i3ZLI-kw3WaIC-3GYuV93b_B",
   "name": "DeepWater-Video1_frame_0012_mask.png"
  },
  {
   "id": "1j6RZ-1sBzVgObz1B1xwjUZKMcoSjpok1",
   "name": "DeepWater-Video1_frame_0011_mask.png"
  },
  {
   "id": "16ZnMuE327GtiWu5v72MIC6rLbtOfouK6",
   "name": "DeepWater-Video1_frame_0010_mask.png"
  },
  {
   "id": "1kbMt7EhMBebsRYE4TMt-BcmM3YyVxBUM",
   "name": "DeepWater-Video1_frame_0009_mask.png"
  },
  {
   "id": "1Wtl4daMwBRyMoxFFEkcd3VZ0sUY6O3Jx",
   "name": "DeepWater-Video1_frame_0008_mask.png"
  },
  {
   "id": "1N-7sfdSxPofZoGKZJdt5DMdvnK5Uz535",
   "name": "DeepWater-Video1_frame_0007_mask.png"
  },
  {
   "id": "14wOJBhlBbO2NTkYe4DcWG2XIGQI_ub9X",
   "name": "DeepWater-Video1_frame_0006_mask.png"
  },
  {
   "id": "11SMgAlHfGrIuWJCaTWiJLPzjFUG9ETn1",
   "name": "DeepWater-Video1_frame_0005_mask.png"
  },
  {
   "id": "1n7Eb_WHLqMgHndfpOmQcQ8QZp6GHVBWg",
   "name": "DeepWater-Video1_frame_0004_mask.png"
  },
  {
   "id": "1T32_MB5kY3hcTknC5ZY9t9fh0tIzHtTS",
   "name": "DeepWater-Video1_frame_0003_mask.png"
  },
  {
   "id": "14WoM_mvyn6HKoQOE5ptqBak160t5Q3WJ",
   "name": "DeepWater-Video1_frame_0002_mask.png"
  },
  {
   "id": "1TBUpJhUM2pF8G6y_b-_6WruxMl7Zq90u",
   "name": "DeepWater-Video1_frame_0001_mask.png"
  },
  {
   "id": "1PH7XodKRHcpM5UvUvHPwHSw6RAvozvfq",
   "name": "DeepWater-Video1_frame_0000_mask.png"
  }
 ],
 "UDCP": [
  {
   "id": "1CztyTjj_JgehMFTGj8KD7Jo2zpMVjaSt",
   "name": "DeepWater-Video7_frame_0009.jpg"
  },
  {
   "id": "1fo_ZGVdYrwje8n4QWkq4voXBxKVCKgOr",
   "name": "DeepWater-Video7_frame_0008.jpg"
  },
  {
   "id": "1GP8paMx_TfVY_11vSR-ZWpZ-bGCgToxb",
   "name": "DeepWater-Video7_frame_0007.jpg"
  },
  {
   "id": "1OWZwRFT10kuZTY7o-5OMkruGAiGQVvN4",
   "name": "DeepWater-Video7_frame_0006.jpg"
  },
  {
   "id": "1WTfgroI2Xfk_HEFWM0866ZB_ZJondjoT",
   "name": "DeepWater-Video7_frame_0005.jpg"
  },
  {
   "id": "1DzMEAsBxZIL8ersXKCtkWe5K3P1UBkA3",
   "name": "DeepWater-Video7_frame_0004.jpg"
  },
  {
   "id": "1Lb0rYzvpBdc-rNmRlTd7a3hH_uwaHZZk",
   "name": "DeepWater-Video7_frame_0003.jpg"
  },
  {
   "id": "1iSWQeYoOcv_ANqNkeJj1aK7oEAw4S1sU",
   "name": "DeepWater-Video7_frame_0002.jpg"
  },
  {
   "id": "1pn6AdhyXHYvsn6dtk9syYoA_tM18qWLv",
   "name": "DeepWater-Video7_frame_0001.jpg"
  },
  {
   "id": "11-17nU6NLSnC-66VHhuJzqAQvSaYs96a",
   "name": "DeepWater-Video7_frame_0000.jpg"
  },
  {
   "id": "1horRE_QCV_dal35zB3pd7RvoaDYnZR6K",
   "name": "DeepWater-Video6_frame_0023.jpg"
  },
  {
   "id": "1nRcbnNV6htFsSe8LPTg7-yINWGtfkKSL",
   "name": "DeepWater-Video6_frame_0022.jpg"
  },
  {
   "id": "1rKcIVZYiCRudvDR51B9Ti7dBP66wA0CY",
   "name": "DeepWater-Video6_frame_0021.jpg"
  },
  {
   "id": "1dsK2PevhtwbDZUWf7s2sDi2soSLDn38y",
   "name": "DeepWater-Video6_frame_0020.jpg"
  },
  {
   "id": "1HArOmW4csknwknLNkS8v6DHc6LbwDDbp",
   "name": "DeepWater-Video6_frame_0019.jpg"
  },
  {
   "id": "1CTmsa97QaNSyaTtKezjbQgzvVsLWL998",
   "name": "DeepWater-Video6_frame_0018.jpg"
  },
  {
   "id": "1vA82p8Pcj0K1KlFadW6n5dnoW7ie8h5q",
   "name": "DeepWater-Video6_frame_0017.jpg"
  },
  {
   "id": "1efEk5pkEvGZ4W_To8edV9rX2iOMJNpmt",
   "name": "DeepWater-Video6_frame_0016.jpg"
  },
  {
   "id": "1MzyvYKXni1yWzSfI5DXkPE2MSy-ZA54A",
   "name": "DeepWater-Video6_frame_0015.jpg"
  },
  {
   "id": "1RgFt45Mt48BSxkqX0o-saVrGB2rTKjOY",
   "name": "DeepWater-Video6_frame_0014.jpg"
  },
  {
   "id": "1wLMjc9KmoFyHUzUgJaTH0EW0QhfVREoy",
   "name": "DeepWater-Video6_frame_0013.jpg"
  },
  {
   "id": "112zUYppkDFW_sUAB0GPP3Hwt8qNrs3zy",
   "name": "DeepWater-Video6_frame_0012.jpg"
  },
  {
   "id": "1AuAqhzNZthZq6rNmshX7Linm0YEMxJsa",
   "name": "DeepWater-Video6_frame_0011.jpg"
  },
  {
   "id": "1sdEhmKDhQOB18Lnsp4GRQvAD6qDKEkT9",
   "name": "DeepWater-Video6_frame_0010.jpg"
  },
  {
   "id": "1wcj9QQMGcOuoNwSFpOyG2WId6bTSyz6S",
   "name": "DeepWater-Video6_frame_0009.jpg"
  },
  {
   "id": "1Ks5m8JXRY9SuH9VX4gWIOy7HcC-m8_1U",
   "name": "DeepWater-Video6_frame_0008.jpg"
  },
  {
   "id": "1Jy5Mv3RGJ8tVi2Qjn5W_JusK8Lz4-5G3",
   "name": "DeepWater-Video6_frame_0007.jpg"
  },
  {
   "id": "15gMOULzzM5joX3ESUgSWyItNC9IjU16c",
   "name": "DeepWater-Video6_frame_0006.jpg"
  },
  {
   "id": "1gcV4OzjoMJXyr_vCiskNqwfmnRAEIMbZ",
   "name": "DeepWater-Video6_frame_0005.jpg"
  },
  {
   "id": "1jCXMWZpyD2Cq9CJo3brsErtGvtZUIbmV",
   "name": "DeepWater-Video6_frame_0004.jpg"
  },
  {
   "id": "1iSq5r1w5pcINveCtKTQpXd3shHxf_8Rw",
   "name": "DeepWater-Video6_frame_0003.jpg"
  },
  {
   "id": "1A35KRNKs1JkryB7j_hIMe0Ub0oF_6942",
   "name": "DeepWater-Video6_frame_0002.jpg"
  },
  {
   "id": "1-2M11eI2---GbRMVXzHO-YW_03ikcO-S",
   "name": "DeepWater-Video6_frame_0001.jpg"
  },
  {
   "id": "1YJSA-HMHtnrazjM4RbHR3F7iJu8gMmbE",
   "name": "DeepWater-Video6_frame_0000.jpg"
  },
  {
   "id": "1vWhRI09znCyYEBZwbwne7KosApkJwAm_",
   "name": "DeepWater-Video3_frame_0048.jpg"
  },
  {
   "id": "1O9w57xNLdYpxyiOh15hua0EV-eUueRJj",
   "name": "DeepWater-Video3_frame_0047.jpg"
  },
  {
   "id": "17uz-6MMUw8ojESqYerH55YbP3bZiqz0k",
   "name": "DeepWater-Video3_frame_0046.jpg"
  },
  {
   "id": "1lr4jF6GgqyE_AXmSdgxlH5v88OZp0mIT",
   "name": "DeepWater-Video3_frame_0045.jpg"
  },
  {
   "id": "1k9A6prGaPrfpnIhLGpC5s7PCObk23vSp",
   "name": "DeepWater-Video3_frame_0044.jpg"
  },
  {
   "id": "1RSY_sc4qxuqEkcd-IpLxtIMkWxTS_rdz",
   "name": "DeepWater-Video3_frame_0043.jpg"
  },
  {
   "id": "1Z2LVXo9pUkKGManTycDjViiz9qYCVVNZ",
   "name": "DeepWater-Video3_frame_0042.jpg"
  },
  {
   "id": "1bvSY62hK_hT0RWz0zyYtGhB9zX3N3pKA",
   "name": "DeepWater-Video3_frame_0041.jpg"
  },
  {
   "id": "1oFv0f3Juok_MHvXaQaF0LQIB5FqubvIo",
   "name": "DeepWater-Video3_frame_0040.jpg"
  },
  {
   "id": "1yCZ0S7EPFzrQfiiTKh9T2NWkxrmuMw_Z",
   "name": "DeepWater-Video3_frame_0039.jpg"
  },
  {
   "id": "1QJTyEV-5C1epUpWkeTfU5gbARyeiB-Cm",
   "name": "DeepWater-Video3_frame_0038.jpg"
  },
  {
   "id": "1TxmGYPuItd_T8JFUdaQbNNyQeYC9kMjM",
   "name": "DeepWater-Video3_frame_0037.jpg"
  },
  {
   "id": "1hKz25csfBB2eUbFcEXYcQcYjmDO7Kktv",
   "name": "DeepWater-Video3_frame_0036.jpg"
  },
  {
   "id": "1GLM9L0BtlUadtsqaFW8M0ILwERnP1TNe",
   "name": "DeepWater-Video3_frame_0035.jpg"
  },
  {
   "id": "15OFpEcY1F8My207KvxZ1GkPPP-_-W2KU",
   "name": "DeepWater-Video3_frame_0034.jpg"
  },
  {
   "id": "18wzURwaCv0K1Sf29dDUt1lt4q6IWW7Qp",
   "name": "DeepWater-Video3_frame_0033.jpg"
  },
  {
   "id": "1wd3jAptcWuPP869nEwHftbIfOqqvqCqZ",
   "name": "DeepWater-Video3_frame_0032.jpg"
  },
  {
   "id": "1mneJi7ewPZyhXGxg89vFZmNxW-AXcx56",
   "name": "DeepWater-Video3_frame_0031.jpg"
  },
  {
   "id": "1XtXFyYd5Jy1rud0mPSwmu722_eYFhowk",
   "name": "DeepWater-Video3_frame_0030.jpg"
  },
  {
   "id": "1zaly5FzTTHrCy7WCQZGzPk0AnsrSvb8P",
   "name": "DeepWater-Video3_frame_0029.jpg"
  },
  {
   "id": "1xhCkkMWVF-oiAU7eOsR4__8vrvYm76FF",
   "name": "DeepWater-Video3_frame_0028.jpg"
  },
  {
   "id": "1rcZG_SxjzUIXOSVezguwZO6qf2ijr86F",
   "name": "DeepWater-Video3_frame_0027.jpg"
  },
  {
   "id": "1ozqsC1pXLQ0GgKteF6_phr5vjw5xXo3E",
   "name": "DeepWater-Video3_frame_0026.jpg"
  },
  {
   "id": "1TfStD2qJ1YxpR1fN8dPH-xKNS0LBN20b",
   "name": "DeepWater-Video3_frame_0025.jpg"
  },
  {
   "id": "1XBDYEhsRa4Y-8Hwk5M6HwJD3rgsOpXQx",
   "name": "DeepWater-Video3_frame_0024.jpg"
  },
  {
   "id": "1zWq-vZc1ama3if5t_ffxoWND57gQ0vu-",
   "name": "DeepWater-Video3_frame_0023.jpg"
  },
  {
   "id": "1euSMQEWYWjyEPsjWGMEibteFtkXsw7qG",
   "name": "DeepWater-Video3_frame_0022.jpg"
  },
  {
   "id": "1pVQWWoNsSB81YSw5c2BpAgqdDuoI8-Xq",
   "name": "DeepWater-Video3_frame_0021.jpg"
  },
  {
   "id": "1XbWTH5V9O_lC_eiSuM_ffxI3zc_nYcLp",
   "name": "DeepWater-Video3_frame_0020.jpg"
  },
  {
   "id": "16OxVSKgRQQ9ATwnPXaM2AMvoyjJ7x9M9",
   "name": "DeepWater-Video3_frame_0019.jpg"
  },
  {
   "id": "1asKDFrNolRt4nO5uTay2K_cAA8JOxAdZ",
   "name": "DeepWater-Video3_frame_0018.jpg"
  },
  {
   "id": "1b6MSTZfi9JXVJOZ740xucmcyZ76SFbVL",
   "name": "DeepWater-Video3_frame_0017.jpg"
  },
  {
   "id": "1a186Cx96Hfx8UFPM2CMEsH7MBZo6uZ9-",
   "name": "DeepWater-Video3_frame_0016.jpg"
  },
  {
   "id": "1qoIXsc8FEdssxrs0d3k5UWSAtUpD6YY8",
   "name": "DeepWater-Video3_frame_0015.jpg"
  },
  {
   "id": "10hpYUE3Hd0482l15YD4Mh0_5EMIfFjmV",
   "name": "DeepWater-Video3_frame_0014.jpg"
  },
  {
   "id": "121uX2gGSYhUoMYxEfaKub4bw_7-mTTOY",
   "name": "DeepWater-Video3_frame_0013.jpg"
  },
  {
   "id": "1VdBsfiGSoFfSepqTEe0586NK0m0DhqD5",
   "name": "DeepWater-Video3_frame_0012.jpg"
  },
  {
   "id": "1e2XvOeFiYCr4yQQEPMHvFHTCLBlC9k0L",
   "name": "DeepWater-Video3_frame_0011.jpg"
  },
  {
   "id": "1X5737gcs3-ryyaXCAaXBqkvxVqBo13V_",
   "name": "DeepWater-Video3_frame_0010.jpg"
  },
  {
   "id": "1_8SGZyBM_DAa4viBICFyJnpZf3-VNqcV",
   "name": "DeepWater-Video3_frame_0009.jpg"
  },
  {
   "id": "1sVK3PDUUCgWDRXyf-lqkFd2z6TjLoH2Q",
   "name": "DeepWater-Video3_frame_0008.jpg"
  },
  {
   "id": "1kgArMbl5cuqtB6dLfYXChWcyTXSeXcuu",
   "name": "DeepWater-Video3_frame_0007.jpg"
  },
  {
   "id": "1yZvhl_U87oRZId3BeZgZK4wdKN4ge4pt",
   "name": "DeepWater-Video3_frame_0006.jpg"
  },
  {
   "id": "1sSphtAujUswp5X1S_1l2ILek7HqSM1G_",
   "name": "DeepWater-Video3_frame_0005.jpg"
  },
  {
   "id": "1XjfOdO9QNNuL4iMkfhxe6SrgvZmX3kd6",
   "name": "DeepWater-Video3_frame_0004.jpg"
  },
  {
   "id": "1jvYoOdJ7Cymsv_VrbbmUZ2QV-HX7NsD5",
   "name": "DeepWater-Video3_frame_0003.jpg"
  },
  {
   "id": "1Uew-o52GDsBQ3KAgAK39WxpwGZpwbBZM",
   "name": "DeepWater-Video3_frame_0002.jpg"
  },
  {
   "id": "1M-5w471lWtWoBwv7gAknzTThvnECl5lo",
   "name": "DeepWater-Video3_frame_0001.jpg"
  },
  {
   "id": "1kHypfJNEAuZb4zm_t1TWNsY97wwG9tJQ",
   "name": "DeepWater-Video3_frame_0000.jpg"
  },
  {
   "id": "16qM4vjRWfSWNgwN7BhqOtdMBveqexBp_",
   "name": "DeepWater-Video1_frame_0021.jpg"
  },
  {
   "id": "1l5aDpcDx8cKAUmwR2XNLHNNI_fzoEP8c",
   "name": "DeepWater-Video1_frame_0020.jpg"
  },
  {
   "id": "1QfrQvaz3YiRtEF0dZYX9KCQXg4MJGpRV",
   "name": "DeepWater-Video1_frame_0019.jpg"
  },
  {
   "id": "1f0RhSjCAhoFq0jKNznvaGj8L033Xk3DC",
   "name": "DeepWater-Video1_frame_0018.jpg"
  },
  {
   "id": "1jfliRcvXEsS5hGcSjg73YGNX1-Xl4p9l",
   "name": "DeepWater-Video1_frame_0017.jpg"
  },
  {
   "id": "1i4xar1M6C0yU6_aicK8_Faajf9ILc4vQ",
   "name": "DeepWater-Video1_frame_0016.jpg"
  },
  {
   "id": "1R3ivGF-pzQz_E4Bbvu-yugIJ7Nv_Y0MQ",
   "name": "DeepWater-Video1_frame_0015.jpg"
  },
  {
   "id": "1LgPORxP9VcIwG7S9kaCa3DxPDaGSXHX_",
   "name": "DeepWater-Video1_frame_0014.jpg"
  },
  {
   "id": "1Wam8BSKug2HjxNx-Gaoe_uwpLl4aVDIO",
   "name": "DeepWater-Video1_frame_0013.jpg"
  },
  {
   "id": "1II9XTjrvIhPPQirz7bvFjiolQex28k42",
   "name": "DeepWater-Video1_frame_0012.jpg"
  },
  {
   "id": "1YrmU4hBwvL9kFse3fthiboZppPs_jC3C",
   "name": "DeepWater-Video1_frame_0011.jpg"
  },
  {
   "id": "1gPV7bYCPju-m1bOIozziO2BNbY2gBUk6",
   "name": "DeepWater-Video1_frame_0010.jpg"
  },
  {
   "id": "1FlIigEynGaE2gtAr9H6QCNgfRdomgigA",
   "name": "DeepWater-Video1_frame_0009.jpg"
  },
  {
   "id": "1cuzMtarrWCFD34JBjZbBjAb7VHQxa-H6",
   "name": "DeepWater-Video1_frame_0008.jpg"
  },
  {
   "id": "1pIFH6fo1-b5jsc-scrWM2JGneyqgKxiF",
   "name": "DeepWater-Video1_frame_0007.jpg"
  },
  {
   "id": "1CCpq5EY2dqwm7LlB6eInwPODuowNzLec",
   "name": "DeepWater-Video1_frame_0006.jpg"
  },
  {
   "id": "16te9-uTg6xf-fnB4No-i2sTCW_BxYLmK",
   "name": "DeepWater-Video1_frame_0005.jpg"
  },
  {
   "id": "1t4_MKIRfSrK1ZMBcNBUfXzZ7ChzaXcz2",
   "name": "DeepWater-Video1_frame_0004.jpg"
  },
  {
   "id": "1MnDBz4iD5hy4wtXa0LZKaE-3jBqKuPDE",
   "name": "DeepWater-Video1_frame_0003.jpg"
  },
  {
   "id": "1q2LkW4GmyQQiI-3Gp-dsWfSUjpwgiSrg",
   "name": "DeepWater-Video1_frame_0002.jpg"
  },
  {
   "id": "1bbORXu4GH36gdVvAzWJ36ZBc0uI7XbZp",
   "name": "DeepWater-Video1_frame_0001.jpg"
  },
  {
   "id": "1GvtqeU8QcvBKI1Afy7h5ArK5hDfIYLXk",
   "name": "DeepWater-Video1_frame_0000.jpg"
  }
 ]
}

In [ ]:
def _drive_service():
    # Authenticated download as a fallback for the anonymous public link.
    # The file is shared correctly ("anyone with the link"),
    # but gdown pulls it anonymously and Drive throttles repeated anonymous
    # hits on the same public link in a row
    # ("have had many accesses"). Signing in with the Colab
    # account gets around it, since the download is no longer anonymous.
    try:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        return build("drive","v3",cache_discovery=False)
    except Exception:
        return None
_HAS_DRIVE_AUTH=_drive_service() is not None

def _download_via_api(file_id,dest):
    import io
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    service=build("drive","v3",cache_discovery=False)  # instancia propia por hilo
    request=service.files().get_media(fileId=file_id)
    buf=io.BytesIO(); downloader=MediaIoBaseDownload(buf,request)
    done=False
    while not done:
        _,done=downloader.next_chunk()
    dest.write_bytes(buf.getvalue())

def download_item(item,retries=4,base_delay=4):
    # This never raises on its own: it returns (folder, name, ok, error) so one
    # bad file doesn't kill the whole check. (Before, an exception inside
    # pool.map stopped the loop halfway and left folders with different frame
    # counts without flagging it here, which then
    # blew up as a confusing AssertionError in the audit cell.)
    folder,f=item
    p=DATA/folder/f["name"]
    p.parent.mkdir(parents=True,exist_ok=True)
    if p.exists() and cv2.imread(str(p),cv2.IMREAD_UNCHANGED) is not None:
        return (folder,f["name"],True,None)
    last_err="descarga vacia o imagen ilegible"
    for attempt in range(1,retries+1):
        try:
            if _HAS_DRIVE_AUTH:
                _download_via_api(f["id"],p)
            else:
                gdown.download(id=f["id"],output=str(p),quiet=True)
            if p.exists() and cv2.imread(str(p),cv2.IMREAD_UNCHANGED) is not None:
                return (folder,f["name"],True,None)
        except Exception as e:
            last_err=str(e).splitlines()[0]
        if attempt<retries:
            time.sleep(base_delay*attempt)
    return (folder,f["name"],False,last_err)

print("Descarga autenticada via API de Drive:" ,"si" if _HAS_DRIVE_AUTH else "no (usando gdown anonimo de respaldo)")
jobs=[(k,f) for k,fs in MANIFEST.items() for f in fs]
results=[]
with ThreadPoolExecutor(max_workers=4 if _HAS_DRIVE_AUTH else 2) as pool:
    for i,res in enumerate(pool.map(download_item,jobs),1):
        results.append(res)
        if not res[2]:
            print(f"  [{i}/{len(jobs)}] fallo {res[0]}/{res[1]}: {res[3]}",flush=True)
        if i%25==0 or i==len(jobs): print(f"Verificados {i}/{len(jobs)}",flush=True)
(OUT/"source_manifest.json").write_text(json.dumps(MANIFEST,indent=2))
failed=[r for r in results if not r[2]]
if failed:
    fail_df=pd.DataFrame(failed,columns=["folder","name","ok","error"]).drop(columns="ok")
    fail_df.to_csv(OUT/"download_failures.csv",index=False)
    detalle="\n".join(f"  - {r[0]}/{r[1]}: {r[3]}" for r in failed[:20])
    raise RuntimeError(
        f"Fallaron {len(failed)}/{len(jobs)} descargas de Drive tras reintentos "
        f"(detalle completo en {OUT/'download_failures.csv'}):\n{detalle}\n"
        "No se continua con un dataset parcial: revisa acceso/cuota de esos archivos en Drive "
        "y vuelve a ejecutar esta celda (los ya descargados no se repiten)."
    )
print("Todas las descargas verificadas correctamente.")


## 2. Audit

Eyeball every mask and confirm the colored region is actually the pipe. Binary masks
(0/1 or 0/255) are fine; multicolor or multilevel ones get rejected. The code can't vouch
for what SAM meant, so if foreground and background come out swapped, flip INVERT_MASKS.
Everything is resized to 64x64 like the earlier prototype, which can wash out fine
structure and stretch non-square frames.

In [ ]:
def key(p): return re.sub(r"_mask$","",Path(p).stem,flags=re.I)
paths={m:{key(p):p for p in (DATA/m).iterdir() if p.is_file()} for m in METHODS}
mask_paths={key(p):p for p in (DATA/"Masks").iterdir() if p.is_file()}
names=sorted(paths["Crudos"])
assert names
_missing_hint=(" Revisa OUT/'download_failures.csv' de la celda de descarga: "
    "probablemente faltan archivos por un fallo de Drive no resuelto.") \
    if (OUT/"download_failures.csv").exists() else ""
for m in METHODS:
    assert set(paths[m])==set(names),f"Frames distintos en {m}; no se acepta comparación parcial.{_missing_hint}"
assert set(mask_paths)<=set(names),"Máscaras sin imagen."
def read_image(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    if a is None: raise ValueError(str(p))
    return cv2.resize(a,(SIZE,SIZE),interpolation=cv2.INTER_AREA).astype(float)/255
def read_mask(p):
    a=cv2.imread(str(p),cv2.IMREAD_UNCHANGED)
    if a is None: raise ValueError(str(p))
    if a.ndim==3:
        if not (np.array_equal(a[...,0],a[...,1]) and np.array_equal(a[...,0],a[...,2])):
            raise ValueError(f"Máscara multicolor: {p}. Revisar.")
        a=a[...,0]
    if not set(np.unique(a)).issubset({0,1,255}): raise ValueError(f"Máscara no binaria: {p}")
    a=(a>0).astype(np.uint8)
    if INVERT_MASKS: a=1-a
    return cv2.resize(a,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)
images={m:{n:read_image(paths[m][n]) for n in names} for m in METHODS}
masks={n:read_mask(p) for n,p in mask_paths.items()}
audit=pd.DataFrame([dict(frame=n,has_mask=n in masks,mask_occupancy=float(masks[n].mean()) if n in masks else np.nan) for n in names])
audit.to_csv(OUT/"dataset_audit.csv",index=False)
display(audit)
fig,axes=plt.subplots(2,4,figsize=(12,6))
for ax,n in zip(axes.flat,sorted(masks)[:8]):
    ax.imshow(images["CLAHE"][n],cmap="gray")
    ax.imshow(np.ma.masked_where(masks[n]==0,masks[n]),cmap="autumn",alpha=.45,vmin=0,vmax=1)
    ax.set_title(n,fontsize=7)
for ax in axes.flat: ax.axis("off")
plt.suptitle("Revisar: región coloreada = estructura SAM")
plt.tight_layout(); plt.savefig(OUT/"mask_audit.png",dpi=160); plt.show()
print("Imágenes:",len(names),"| con máscara:",len(masks),"| solo inferencia:",len(names)-len(masks))


## 3. Split and patches

With three or more labeled videos we split whole videos apart. With fewer, we fall back
to ordered 60/20/20 time blocks and drop one frame between blocks so neighbors don't leak
across the split. If all the masks come from a single video, treat the numbers as
exploratory within that video, they say nothing about generalizing to new ones. Same
images, coordinates and labels go to every model, and we don't rebalance or subsample the
test set.

In [ ]:
def video(n): return n.split("_frame_")[0]
labeled=sorted(masks)
vids=sorted({video(n) for n in labeled})
split={}
if len(vids)>=3:
    order=list(np.random.default_rng(42).permutation(vids))
    k=max(1,int(len(vids)*.2))
    for n in labeled:
        split[n]="test" if video(n) in order[:k] else "val" if video(n) in order[k:2*k] else "train"
    split_note="Separación por videos completos."
else:
    for v in vids:
        seq=sorted(n for n in labeled if video(n)==v)
        if len(seq)<10: raise ValueError("Se requieren al menos 10 frames por video para bloques temporales.")
        a,b=int(.6*len(seq)),int(.8*len(seq))
        for i,n in enumerate(seq):
            split[n]="gap" if i in (a,b) else "train" if i<a else "val" if i<b else "test"
    split_note="Exploratorio: bloques temporales y embargo de 1 frame; no evaluación en videos nuevos."
display(Markdown("**"+split_note+"**"))
pd.DataFrame([dict(frame=n,video=video(n),split=s) for n,s in split.items()]).to_csv(OUT/"split_manifest.csv",index=False)
coords=[(r,c) for r in range(0,SIZE-PATCH+1,PATCH) for c in range(0,SIZE-PATCH+1,PATCH)]
records=[]
for n in labeled:
    for r,c in coords:
        occ=float(masks[n][r:r+PATCH,c:c+PATCH].mean())
        label=1 if occ>=STRUCTURE_MIN else 0 if occ<=BACKGROUND_MAX else -1
        records.append(dict(frame=n,row=r,col=c,occupancy=occ,y=label,split=split[n]))
meta=pd.DataFrame(records)
meta.to_csv(OUT/"patch_manifest.csv",index=False)
idx={s:np.flatnonzero((meta.split==s)&(meta.y>=0)) for s in ["train","val","test"]}
y=meta.y.to_numpy()
for s,ii in idx.items():
    assert set(y[ii])=={0,1},f"{s} no contiene ambas clases. Revisar máscaras; no cambiar según rendimiento test."
display(pd.crosstab(meta.split,meta.y).rename(columns={-1:"ambiguo",0:"fondo",1:"estructura"}))


In [ ]:
def amplitudes(a):
    a=np.asarray(a,dtype=float).reshape(-1,64).copy()
    norm=np.linalg.norm(a,axis=1,keepdims=True)
    zero=norm[:,0]<1e-12
    a[zero]=1/8; norm[zero]=1
    return a/norm
X={}
for m in METHODS:
    X[m]=amplitudes([images[m][z.frame][z.row:z.row+PATCH,z.col:z.col+PATCH] for z in meta.itertuples()])
    assert np.allclose(np.linalg.norm(X[m],axis=1),1)
# L2 drops global brightness; proportional patches end up with the same amplitudes.


## 4. Training

VQC: 64 amplitudes, 6 qubits, two StronglyEntanglingLayers, output p(structure)=(1+<Z>)/2.
Adam with a weighted binary loss, minibatches, and every training patch seen each epoch.
Checkpoint on validation F1. The SVM uses the same L2 vectors and picks C on validation.
Three seeds for the VQC. The test set never decides the filter, the epoch or any
hyperparameter. CPU is enough for this simulator.

In [ ]:
dev=qml.device("default.qubit",wires=6,shots=None)
@qml.qnode(dev,interface="torch",diff_method="backprop")
def circuit(x,w):
    qml.AmplitudeEmbedding(x,wires=range(6),normalize=False)
    qml.StronglyEntanglingLayers(w,wires=range(6))
    return qml.expval(qml.PauliZ(0))
def probability(x,w): return ((1+circuit(x,w))/2).clamp(1e-7,1-1e-7)
def predict_vqc(x,w):
    with torch.no_grad():
        return np.concatenate([probability(torch.tensor(x[i:i+BATCH]),w).numpy().reshape(-1) for i in range(0,len(x),BATCH)])
def metrics(yy,pp):
    return dict(accuracy=accuracy_score(yy,pp),balanced_accuracy=balanced_accuracy_score(yy,pp),
        precision=precision_score(yy,pp,zero_division=0),recall=recall_score(yy,pp,zero_division=0),
        f1=f1_score(yy,pp,zero_division=0),iou=jaccard_score(yy,pp,zero_division=0))
def train_vqc(x,seed):
    torch.manual_seed(seed)
    w=torch.nn.Parameter(.1*torch.randn(2,6,3))
    opt=torch.optim.Adam([w],lr=LR)
    xx=torch.tensor(x[idx["train"]]); yy=torch.tensor(y[idx["train"]],dtype=torch.float64)
    cw=torch.tensor(len(yy)/(2*np.bincount(y[idx["train"]],minlength=2)))
    best=-1; hist=[]
    for epoch in range(1,EPOCHS+1):
        total=0.
        for batch in torch.randperm(len(xx)).split(BATCH):
            opt.zero_grad(); p=probability(xx[batch],w)
            loss=(-(yy[batch]*torch.log(p)+(1-yy[batch])*torch.log(1-p))*cw[yy[batch].long()]).mean()
            loss.backward()
            assert w.grad is not None and torch.isfinite(w.grad).all()
            opt.step(); total+=loss.item()*len(batch)
        vf=f1_score(y[idx["val"]],predict_vqc(x[idx["val"]],w)>=.5,zero_division=0)
        hist.append(dict(epoch=epoch,loss=total/len(xx),val_f1=vf))
        if vf>best: best=vf; best_w=w.detach().clone()
        if epoch==1 or epoch%5==0: print(f"seed={seed} epoch={epoch}/{EPOCHS} loss={total/len(xx):.4f} val_F1={vf:.3f}",flush=True)
    return best_w,hist
test_w=torch.nn.Parameter(torch.randn(2,6,3)*.1)
probability(torch.tensor(X["Crudos"][:2]),test_w).sum().backward()
assert test_w.grad is not None and torch.isfinite(test_w.grad).all()
print("Gradiente/batching OK")


In [ ]:
models={}; histories={}; rows=[]; pred_rows=[]
for m in METHODS:
    print("\nENTRADA:",m,flush=True); models[m]={}
    for seed in SEEDS:
        start=time.perf_counter()
        w,h=train_vqc(X[m],seed)
        elapsed=time.perf_counter()-start
        tag=f"VQC_{seed}"; models[m][tag]=w; histories[(m,tag)]=h
        np.save(OUT/f"{m}_{tag}_weights.npy",w.numpy())
        for s,ii in idx.items():
            prob=predict_vqc(X[m][ii],w); pp=(prob>=.5).astype(int)
            rows.append(dict(method=m,model="VQC",seed=seed,split=s,seconds=elapsed,**metrics(y[ii],pp)))
            for j,p,score in zip(ii,pp,prob):
                pred_rows.append(dict(method=m,model=tag,patch_index=int(j),split=s,y=int(y[j]),prediction=int(p),score=float(score)))
    best=-1
    for C in [.1,1,10]:
        svm=SVC(C=C,kernel="rbf",gamma="scale",class_weight="balanced")
        svm.fit(X[m][idx["train"]],y[idx["train"]])
        vf=f1_score(y[idx["val"]],svm.predict(X[m][idx["val"]]),zero_division=0)
        if vf>best: best=vf; best_svm=svm
    models[m]["SVM"]=best_svm
    joblib.dump(best_svm,OUT/f"{m}_SVM.joblib")
    for s,ii in idx.items():
        pp=best_svm.predict(X[m][ii])
        rows.append(dict(method=m,model="SVM",seed=-1,split=s,seconds=np.nan,**metrics(y[ii],pp)))
        for j,p in zip(ii,pp):
            pred_rows.append(dict(method=m,model="SVM",patch_index=int(j),split=s,y=int(y[j]),prediction=int(p),score=np.nan))
    pd.DataFrame(rows).to_csv(OUT/"metrics.csv",index=False)
results=pd.DataFrame(rows); predictions=pd.DataFrame(pred_rows)
predictions.to_csv(OUT/"patch_predictions.csv",index=False)
pd.concat([pd.DataFrame(h).assign(method=m,model=t) for (m,t),h in histories.items()]).to_csv(OUT/"learning_curves.csv",index=False)


## 5. Presentation deliverable

Training accuracy and the VQC confusion matrices. F1 and IoU here are for patch
classification, not Sprint 2 edges or pixel-level segmentation. The mean and spread across
seeds is not a generalization confidence interval, don't read it as one.

In [ ]:
val=results[(results.split=="val")&(results.model=="VQC")]
chosen_method=val.groupby("method").f1.mean().idxmax()
def chosen_seed_for(m):
    return int(val[val.method==m].sort_values(["f1","seed"],ascending=[False,True]).iloc[0].seed)
chosen_seed=chosen_seed_for(chosen_method); chosen_tag=f"VQC_{chosen_seed}"
display(Markdown(f"### Filtro seleccionado por validación: {chosen_method}\n{split_note}"))
summary=results.groupby(["method","model","split"])[["accuracy","balanced_accuracy","precision","recall","f1","iou"]].agg(["mean","std"])
display(summary.round(4)); summary.to_csv(OUT/"comparison_summary.csv")
fig,axes=plt.subplots(2,4,figsize=(15,7))
for col,m in enumerate(METHODS):
    for row,tag in enumerate([f"VQC_{chosen_seed_for(m)}","SVM"]):
        p=predictions[(predictions.method==m)&(predictions.model==tag)&(predictions.split=="test")]
        cm=confusion_matrix(p.y,p.prediction,labels=[0,1])
        np.save(OUT/f"{m}_{tag}_confusion_test.npy",cm)
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,ax=axes[row,col],
            xticklabels=["Fondo","Estructura"],yticklabels=["Fondo","Estructura"])
        axes[row,col].set(title=f"{m} · {tag}",xlabel="Predicción",ylabel="Referencia SAM")
plt.tight_layout(); plt.savefig(OUT/"confusion_matrices.png",dpi=180); plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
results[results.split=="test"].groupby(["method","model"]).f1.mean().unstack().plot.bar(ax=axes[0],ylim=(0,1),rot=0)
axes[0].set_title("F1 test — promedio semillas VQC")
for m in METHODS:
    h=pd.DataFrame(histories[(m,f"VQC_{SEEDS[0]}")]); axes[1].plot(h.epoch,h.val_f1,label=m)
axes[1].legend(); axes[1].set(title="F1 validación durante entrenamiento",xlabel="Época")
plt.tight_layout(); plt.savefig(OUT/"comparison_and_training.png",dpi=180); plt.show()
selected=results[(results.method==chosen_method)&(results.model=="VQC")&(results.seed==chosen_seed)]
display(Markdown("### Task 3.1: accuracy entrenamiento y matriz VQC"))
display(selected[["split","accuracy","precision","recall","f1"]].round(4))
p=predictions[(predictions.method==chosen_method)&(predictions.model==chosen_tag)&(predictions.split=="test")]
fig,ax=plt.subplots(figsize=(5,4))
sns.heatmap(confusion_matrix(p.y,p.prediction,labels=[0,1]),annot=True,fmt="d",cmap="Purples",
    xticklabels=["Fondo","Estructura"],yticklabels=["Fondo","Estructura"],ax=ax)
ax.set(xlabel="Predicción",ylabel="Referencia SAM",title=f"VQC · {chosen_method} · test")
plt.tight_layout(); plt.savefig(OUT/"VQC_deliverable_confusion.png",dpi=200); plt.show()


## 6. Spatial predictions for every frame

Each filter uses its own validation-picked VQC and SVM. The maps are 8x8 blocks, so they're
coarse, not fine contours. Frames without a mask are flagged as reference-free inference and
we don't compute accuracy on them.

In [ ]:
inference=[]; maps={}
for m in METHODS:
    maps[m]={}; dest=OUT/"spatial_maps"/m; dest.mkdir(parents=True,exist_ok=True)
    for n in names:
        a=images[m][n]; xx=amplitudes([a[r:r+PATCH,c:c+PATCH] for r,c in coords])
        pp=predict_vqc(xx,models[m][f"VQC_{chosen_seed_for(m)}"])>=.5
        sv=models[m]["SVM"].predict(xx)
        predmap=np.zeros((SIZE,SIZE),dtype=np.uint8); svmmap=np.zeros_like(predmap)
        for (r,c),v,s in zip(coords,pp,sv):
            predmap[r:r+PATCH,c:c+PATCH]=int(v); svmmap[r:r+PATCH,c:c+PATCH]=int(s)
        maps[m][n]=predmap
        cv2.imwrite(str(dest/f"{n}_VQC.png"),predmap*255)
        cv2.imwrite(str(dest/f"{n}_SVM.png"),svmmap*255)
        inference.append(dict(method=m,frame=n,usage=split.get(n,"unlabeled_inference"),
            has_mask=n in masks,vqc_structure_fraction=float(pp.mean()),svm_structure_fraction=float(sv.mean())))
    print("Mapas completos:",m,len(names),flush=True)
pd.DataFrame(inference).to_csv(OUT/"all_frames_inference.csv",index=False)
examples=[n for n in names if split.get(n)=="test"][:2]+[n for n in names if n not in masks][:2]
fig,axes=plt.subplots(len(examples),3,figsize=(10,3*len(examples)),squeeze=False)
for row,n in enumerate(examples):
    axes[row,0].imshow(images[chosen_method][n],cmap="gray"); axes[row,0].set_title(n,fontsize=8)
    axes[row,1].imshow(masks[n] if n in masks else np.zeros((SIZE,SIZE)),cmap="gray",vmin=0,vmax=1)
    axes[row,1].set_title("Referencia SAM" if n in masks else "SIN referencia: solo inferencia")
    axes[row,2].imshow(maps[chosen_method][n],cmap="gray",vmin=0,vmax=1); axes[row,2].set_title("VQC: bloques")
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig(OUT/"spatial_examples.png",dpi=180); plt.show()


In [ ]:
config=dict(size=SIZE,patch=PATCH,seeds=SEEDS,epochs=EPOCHS,batch=BATCH,lr=LR,
    selected_method=chosen_method,selected_seed=chosen_seed,split_note=split_note,
    background_max=BACKGROUND_MAX,structure_min=STRUCTURE_MIN,invert_masks=INVERT_MASKS,
    mask_status="SAM pseudo-labels; human semantic review not certified")
(OUT/"config.json").write_text(json.dumps(config,indent=2))
(OUT/"requirements_executed.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
report=f"""# Task 3.1 — Quantum Patch Classifier
VQC: amplitude encoding, 6 qubits, 2 capas. Comparador: SVM-RBF clásico, no QSVM.
Entradas: crudos, CLAHE, MSRCR, UDCP. Mismas etiquetas/coordenadas/partición.
Filtro seleccionado por validación: {chosen_method}; semilla mostrada: {chosen_seed}.
{split_note}
Máscaras SAM: requieren revisión semántica. Sin máscaras: solo inferencia.
No se demuestra ventaja cuántica con estos resultados.
Accuracy del VQC seleccionado:
{selected[["split","accuracy","f1"]].to_string(index=False)}
Métricas completas: metrics.csv / comparison_summary.csv.
Matriz de exposición: VQC_deliverable_confusion.png.
Mapas de todos los frames: spatial_maps/.

## Save the executed deliverable to GitHub
Colab: Archivo → Guardar una copia en GitHub.
Seleccionar tu repositorio, rama feat/quantum-classification.
Nombre exacto: 4_quantum_patch_classifier.ipynb.
Subir resultados si corresponde. No publicar datos originales sin permiso.
No se publicó automáticamente: no se proporcionó repositorio.
"""
(OUT/"README.md").write_text(report)
display(Markdown(report))
archive=shutil.make_archive(str(OUT),"zip",OUT)
print("Resultados:",archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Fuera de Colab: usa la ruta anterior.")


## References

PennyLane. *AmplitudeEmbedding*. https://docs.pennylane.ai/en/stable/code/api/pennylane.AmplitudeEmbedding.html

PennyLane. *PyTorch interface*. https://docs.pennylane.ai/en/stable/introduction/interfaces/torch.html

This notebook uses a VQC, one of the options Task 3.1 allows. The numbers come out of
running it, there are no preloaded metrics.